# Rol 3 — Operadores continuos, Días 1 y 2

Se conservan las comprobaciones del torneo del Día 1. El Día 2 incorpora SUS, BLX-alpha, cruce uniforme, mutación gaussiana y no uniforme. Se revisaron el examen y `ag_ejemplo_reinas.ipynb` del docente: se conservan funciones independientes, selección por aptitud y copias; OX y swap para permutaciones se sustituyen por operadores adecuados a genes continuos. No se ejecutan experimentos del Día 3.


In [1]:
from pathlib import Path
import sys
import numpy as np

RAIZ = Path.cwd().resolve()
if not (RAIZ / "src" / "representacion.py").is_file():
    RAIZ = RAIZ.parent
if not (RAIZ / "src" / "representacion.py").is_file():
    raise FileNotFoundError("Ejecuta desde la raiz del repositorio o notebooks/.")
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src.operadores import seleccion_torneo
from src.representacion import (
    num_genes, generar_poblacion_inicial, validar_individuo,
)

D = num_genes()
print(f"Genes actuales: {D}")

Genes actuales: 8


## 1. Maximizacion y copia independiente

Las aptitudes de este ejemplo son artificiales para aislar el operador.
Tambien se admiten aptitudes negativas: solo importa su orden.
Con N = k = 3 participan todos y siempre debe ganar el maximo.

In [2]:
poblacion = np.array([np.full(D, valor) for valor in (0.2, 0.5, 0.8)])
aptitudes = np.array([-3.0, 2.0, -1.0])
poblacion_original = poblacion.copy()
aptitudes_originales = aptitudes.copy()
rng = np.random.default_rng(42)

for _ in range(20):
    ganador = seleccion_torneo(poblacion, aptitudes, k=3, rng=rng)
    assert np.array_equal(ganador, poblacion[1])
    assert not np.shares_memory(ganador, poblacion)

assert np.array_equal(poblacion, poblacion_original)
assert np.array_equal(aptitudes, aptitudes_originales)
ganador[0] = 0.0
assert np.array_equal(poblacion, poblacion_original)
assert np.array_equal(aptitudes, aptitudes_originales)
print("Maximo en 20 torneos, entradas intactas y copia independiente: OK")

Maximo en 20 torneos, entradas intactas y copia independiente: OK


## 2. Reproducibilidad, empates y compatibilidad

El generador se crea una sola vez por secuencia y se comparte entre llamadas.
Dos generadores con igual semilla y las mismas operaciones reproducen el sorteo.
La prueba de empates comprueba el orden sorteado, no el indice de la poblacion.

In [3]:
poblacion_aleatoria = generar_poblacion_inicial(10, rng=np.random.default_rng(2024))
aptitudes_ejemplo = np.arange(10, dtype=float)
rng_a = np.random.default_rng(7)
rng_b = np.random.default_rng(7)
secuencia_a = [seleccion_torneo(poblacion_aleatoria, aptitudes_ejemplo, rng=rng_a) for _ in range(20)]
secuencia_b = [seleccion_torneo(poblacion_aleatoria, aptitudes_ejemplo, rng=rng_b) for _ in range(20)]
assert np.array_equal(secuencia_a, secuencia_b)

# Contrastar el sorteo y su avance con un generador independiente.
rng_control = np.random.default_rng(7)
for ganador in secuencia_a:
    indices = rng_control.choice(10, size=3, replace=False)
    assert len(np.unique(indices)) == 3
    assert np.array_equal(ganador, poblacion_aleatoria[indices[np.argmax(aptitudes_ejemplo[indices])]])
    valido, mensaje = validar_individuo(ganador)
    assert valido, mensaje
    assert ganador.shape == (D,) and ganador.dtype == np.float64

rng_empate = np.random.default_rng(101)
rng_control = np.random.default_rng(101)
indices = rng_control.choice(10, size=3, replace=False)
ganador_empate = seleccion_torneo(poblacion_aleatoria, np.zeros(10), rng=rng_empate)
assert np.array_equal(ganador_empate, poblacion_aleatoria[indices[0]])

# k=1, entero NumPy y generador omitido tambien son validos.
ganador_unico = seleccion_torneo(poblacion[:1], [-5.0], k=np.int64(1))
assert np.array_equal(ganador_unico, poblacion[0])
print("Reproducibilidad, muestreo, empates, k=1 y compatibilidad float64: OK")

Reproducibilidad, muestreo, empates, k=1 y compatibilidad float64: OK


## 3. Entradas invalidas

Se informa ValueError sin reducir k ni reparar genes. La correspondencia
semantica entre fitness[i] y poblacion[i] es responsabilidad del llamador;
el operador puede comprobar la longitud, pero no detectar una permutacion
incorrecta de aptitudes ya calculadas.

In [4]:
def comprobar_value_error(pob, fits, k=3):
    try:
        seleccion_torneo(pob, fits, k=k, rng=np.random.default_rng(42))
    except ValueError:
        return
    raise AssertionError("Se esperaba ValueError")

casos_invalidos = [
    (poblacion[:2], aptitudes[:2], 3),
    (poblacion, aptitudes[:2], 3),
    (poblacion, aptitudes.reshape(3, 1), 3),
    (poblacion[0], aptitudes, 3),
    (np.empty((0, D)), np.array([]), 1),
    (np.zeros((3, D + 1)), aptitudes, 3),
]
for valor in (np.nan, np.inf, -np.inf):
    fits_invalidos = aptitudes.copy()
    fits_invalidos[0] = valor
    casos_invalidos.append((poblacion, fits_invalidos, 3))
for valor in (-0.01, 1.01, np.nan, np.inf):
    pob_invalida = poblacion.copy()
    pob_invalida[0, 0] = valor
    casos_invalidos.append((pob_invalida, aptitudes, 3))
for k_invalido in (0, -1, 4, 2.5, 3.0, True, np.bool_(False)):
    casos_invalidos.append((poblacion, aptitudes, k_invalido))
for pob, fits, k in casos_invalidos:
    comprobar_value_error(pob, fits, k)
print(f"Rechazo de {len(casos_invalidos)} casos invalidos: OK")

Rechazo de 20 casos invalidos: OK


## 4. Integracion real con los Roles 1 y 2

Se reutiliza `cargar_melodia_csv()` y la lectura `sf.read(..., dtype="float32")`
que emplea el Rol 2 para evaluar exactamente el WAV guardado. Su notebook
prepara ese montaje con muestras reales, las fronteras del CSV, un suavizado
de 5 ms y ganancia global. Aqui se consume el montaje existente, sin repetir
sus descargas ni escribir WAV o resultados experimentales.

El JSON debe coincidir con las notas y duraciones del CSV. No se recorta,
remuestrea ni normaliza la referencia. Si faltan archivos o dependencias,
la celda informa el problema y se detiene: no usa una referencia sustituta.
Las tres aptitudes se calculan de nuevo con `evaluar_individuo()`; no se leen
resultados guardados por el Rol 2. Ambas metricas se maximizan; nunca pasar
`error_stft_db` directamente al torneo.

In [5]:
import json
try:
    import soundfile as sf
    from src.extraccion import cargar_melodia_csv
    from src.sintesis import FRECUENCIA_MUESTREO
    from src.fitness import evaluar_individuo
except ImportError as error:
    raise ImportError("Integracion no ejecutada: faltan dependencias de requirements.txt.") from error

ruta_csv = RAIZ / "datos" / "procesados" / "melodia_01.csv"
ruta_ref = RAIZ / "datos" / "audio" / "referencias" / "flauta_referencia_01.wav"
ruta_json = ruta_ref.with_suffix(".json")
for ruta in (ruta_csv, ruta_ref, ruta_json):
    if not ruta.is_file():
        raise FileNotFoundError(f"Integracion no ejecutada; falta: {ruta.name}")

notas, duraciones = cargar_melodia_csv(ruta_csv)
metadatos = json.loads(ruta_json.read_text(encoding="utf-8"))
assert np.array_equal(notas, metadatos["notas"]), "Notas distintas del montaje"
assert np.array_equal(duraciones, metadatos["duraciones_s"]), "Duraciones distintas del montaje"
referencia, sr_ref = sf.read(ruta_ref, dtype="float32")
assert sr_ref == FRECUENCIA_MUESTREO
assert referencia.ndim == 1
assert sf.info(ruta_ref).subtype == "PCM_16"
assert referencia.size == int(round(duraciones.sum() * FRECUENCIA_MUESTREO))
assert np.all(np.isfinite(referencia)) and np.max(np.abs(referencia)) > 1e-8

rng_integracion = np.random.default_rng(2026)
poblacion_real = generar_poblacion_inicial(3, rng=rng_integracion)
aptitudes_reales = np.array([
    evaluar_individuo(gen, notas, duraciones, referencia, metrica="stft")
    for gen in poblacion_real
])
assert aptitudes_reales.shape == (3,) and np.all(np.isfinite(aptitudes_reales))
ganador_real = seleccion_torneo(poblacion_real, aptitudes_reales, k=3, rng=rng_integracion)
assert np.array_equal(ganador_real, poblacion_real[np.argmax(aptitudes_reales)])
valido, mensaje = validar_individuo(ganador_real)
assert valido, mensaje
assert ganador_real.shape == (D,) and ganador_real.dtype == np.float64
assert not np.shares_memory(ganador_real, poblacion_real)
print("Referencia:", referencia.size, "muestras a", sr_ref, "Hz")
print("Aptitudes STFT calculadas:", aptitudes_reales)
print("Indice del ganador:", int(np.argmax(aptitudes_reales)))
print("Integracion real: OK. No se escribieron audios ni resultados experimentales.")

Referencia: 59645 muestras a 22050 Hz
Aptitudes STFT calculadas: [0.67976701 0.49070326 0.70833249]
Indice del ganador: 2
Integracion real: OK. No se escribieron audios ni resultados experimentales.


## 5. Contrato conservado del torneo

`padre = seleccion_torneo(poblacion, fitness, k=3, rng=rng)` devuelve solo el cromosoma. La población y sus aptitudes conservan el mismo orden.


## 6. Selección SUS por lote

SUS utiliza un único inicio aleatorio y punteros equidistantes, con pesos
proporcionales a aptitudes no negativas. Puede repetir individuos. Si todas
las aptitudes son cero usa pesos iguales; escala por el máximo antes de sumar.
No son ruletas independientes. Torneo acepta negativos; SUS los rechaza.

In [6]:
from src.operadores import (
    seleccion_sus, cruce_blend, cruce_uniforme,
    mutacion_gaussiana, mutacion_no_uniforme,
)

rng_operadores = np.random.default_rng(42)
lote = seleccion_sus(poblacion_real, aptitudes_reales, cantidad=4, rng=rng_operadores)
assert lote.shape == (4, D)
assert not np.shares_memory(lote, poblacion_real)
assert all(validar_individuo(g)[0] for g in lote)
print("SUS: lote válido de", len(lote), "padres")

SUS: lote válido de 4 padres


## 7. Cruces y mutaciones

BLX-alpha muestrea cada hijo por gen en el intervalo parental ampliado por
alpha veces su distancia. El uniforme intercambia genes con máscara 0.50 y
herencia complementaria. Ambos devuelven dos hijos; no aplican p_cruce.

La gaussiana suma ruido N(0,sigma) a los genes elegidos. La no uniforme elige
dirección al límite y usa delta = y*(1-r**((1-t/T)**b)). Su desplazamiento se
reduce con t y es cero en t=T. Ambas aplican p_mut por gen, recortan y validan.
Seleccionar un gen no implica necesariamente cambiarlo (sigma=0, clipping o t=T).
No se redondean genes. Las entradas inválidas se rechazan, no se reparan.

In [7]:
padre1, padre2 = lote[:2]
originales = lote.copy()
h_blx = cruce_blend(padre1, padre2, alpha=0.5, rng=rng_operadores)
h_uniformes = cruce_uniforme(padre1, padre2, rng=rng_operadores)
assert np.all((h_uniformes[0] == padre1) | (h_uniformes[0] == padre2))
assert np.all((h_uniformes[0] == padre1) & (h_uniformes[1] == padre2)
              | (h_uniformes[0] == padre2) & (h_uniformes[1] == padre1))
m_gauss = mutacion_gaussiana(h_blx[0], p_mut=0.05, sigma=0.08, rng=rng_operadores)
m_no_uniforme = mutacion_no_uniforme(h_blx[1], 0, 50, p_mut=0.05, b=2.0, rng=rng_operadores)
for hijo in (*h_blx, *h_uniformes, m_gauss, m_no_uniforme):
    assert validar_individuo(hijo)[0] and hijo.dtype == np.float64
    assert not np.shares_memory(hijo, lote)
np.testing.assert_array_equal(lote, originales)
np.testing.assert_array_equal(mutacion_gaussiana(padre1, p_mut=0), padre1)
np.testing.assert_array_equal(mutacion_no_uniforme(padre1, 50, 50, p_mut=1), padre1)
print("Cruces y mutaciones: validez, herencia y copias comprobadas")

Cruces y mutaciones: validez, herencia y copias comprobadas


## 8. Ejemplo ejecutable de reproducción para el Rol 4

El módulo actual no tiene ejecutar_ag(): esto es **un paso de reproducción**,
no una corrida completa ni una comparación de combinaciones. Usa inicialización,
evaluación, elitismo y estadísticas existentes. Se evalúan individuos nuevos
contra el WAV real cargado arriba; no se leen resultados del Rol 2 ni se escriben audios.

Con N=5 y elitismo=2 quedan tres plazas. Se solicita un lote SUS de cuatro
padres, se usan dos parejas y se conserva solo el primer hijo de la última
pareja. p_cruce=0.90 se decide una vez por pareja; si no ocurre, se copian los
padres. Cada hijo conservado se muta sin filtro previo por individuo.

Mapeo temporal usado: la población evaluada g crea g+1 con t=g y T=G.
Por tanto g=0,...,G-1 durante la reproducción; t=T es el extremo explícito de
desplazamiento cero, no un reemplazo adicional. Es la convención de integración
de este ejemplo, entregada al Rol 4. La semilla de síntesis permanece independiente.

In [8]:
from src.configuracion import ConfiguracionAG
from src.algoritmo_genetico import (
    inicializar_poblacion, evaluar_poblacion,
    seleccionar_elites, calcular_estadisticas,
)

config = ConfiguracionAG(tam_poblacion=5, elitismo=2, semilla=2026)
rng = np.random.default_rng(config.semilla)
P = inicializar_poblacion(config, rng=rng)
evaluaciones = 0

def aptitud_contada(gen):
    global evaluaciones
    evaluaciones += 1
    return evaluar_individuo(gen, notas, duraciones, referencia, metrica="stft")

F = evaluar_poblacion(P, aptitud_contada)
P_original, F_original = P.copy(), F.copy()

# Alternativa de selección para una pareja: dos torneos con el mismo RNG.
# No se ejecuta aquí para no consumir sorteos adicionales:
# p1 = seleccion_torneo(P, F, k=config.tam_torneo, rng=rng)
# p2 = seleccion_torneo(P, F, k=config.tam_torneo, rng=rng)

elites = seleccionar_elites(P, F, config.elitismo)
plazas = config.tam_poblacion - len(elites)
cantidad_padres = 2 * ((plazas + 1) // 2)
padres = seleccion_sus(P, F, cantidad=cantidad_padres, rng=rng)
nueva = [elite.copy() for elite in elites]
generacion = 0
for i in range(0, cantidad_padres, 2):
    p1, p2 = padres[i:i+2]
    if rng.random() < config.probabilidad_cruce:
        hijos = cruce_blend(p1, p2, alpha=0.5, rng=rng)
    else:
        hijos = (p1.copy(), p2.copy())
    for hijo in hijos:
        if len(nueva) == config.tam_poblacion:
            break
        hijo = mutacion_no_uniforme(
            hijo, generacion, config.max_generaciones,
            p_mut=config.probabilidad_mutacion, b=2.0, rng=rng,
        )
        assert validar_individuo(hijo)[0]
        nueva.append(hijo.copy())

P_nueva = np.array(nueva, dtype=np.float64)
F_nueva = evaluar_poblacion(P_nueva, aptitud_contada)
estadisticas = calcular_estadisticas(P_nueva, F_nueva, generacion=1)
assert P_nueva.shape == P.shape
assert not np.shares_memory(P_nueva, P)
np.testing.assert_array_equal(P, P_original)
np.testing.assert_array_equal(F, F_original)
np.testing.assert_array_equal(P_nueva[:config.elitismo], elites)
assert evaluaciones == 10  # 5 iniciales y 5 de nueva población, incluidas élites.
assert F_nueva.max() >= F.max()
print("Reproducción de prueba: 3 plazas, 4 padres SUS; tamaño y élites conservados")
print("Evaluaciones reales de este ejemplo:", evaluaciones)
print("Aptitudes de los hijos conservados:", F_nueva[config.elitismo:])
print("Estadísticas de la población de prueba:", estadisticas)

Reproducción de prueba: 3 plazas, 4 padres SUS; tamaño y élites conservados
Evaluaciones reales de este ejemplo: 10
Aptitudes de los hijos conservados: [0.70650975 0.70705436 0.72033843]
Estadísticas de la población de prueba: EstadisticasGeneracion(generacion=1, fitness_maximo=0.7203384257807997, fitness_medio=0.7044004090331718, desviacion_fitness=0.013325005721359206, diversidad_genetica=0.16063945957893583)


## 9. Estado de integración

Operadores del Día 2 disponibles y reproducción comprobable con las interfaces
reales. La conexión al ciclo evolutivo completo pertenece al Rol 4 y permanece
pendiente porque no existe ejecutar_ag() en la base revisada. No se ejecutan
las tres combinaciones, doce configuraciones ni réplicas del Día 3.
Las firmas y reglas definitivas están en `documentacion/contratos/interfaz_operadores_rol3.md`.